## 5. Feature Engineering

Invoice-to-Payment Process Optimization — Extension

*See the [README](#) for full project overview, business problem, and dataset details.*


This notebook builds the feature tables for two predictive analyses, whose
models are trained and evaluated in Notebook 6:

**Part 1 — Case-Level Throughput Prediction:**

- **Target (continuous):** `gr_to_clear_days` (similar to [Rząd, Wojnecka, Rutkowski & Guliński (2019)](https://icpmconference.org/2019/wp-content/uploads/sites/6/2019/07/BPI-Challenge-Submission-2.pdf))
- **Prediction point:** as-of-Goods-Receipt — all features reflect only
  information available at or before each case's own GR timestamp (avoids
  hindsight leakage; see 5.2.1 discussion)
- **Sample:** `companyID_0000`, both 3-way match categories only (invoice
  before/after GR) — 234,748 cases total, **181,552 finished, cleaned
  cases** used for modeling
- **Features:** item category, spend area, order value, document type,
  exception-activity flags (adapted from Rząd et al.'s "Undesired
  Activities" taxonomy — Table 2 — filtered per-case to only count
  occurrences at or before GR), vendor tier (Gold/Silver/Bronze/No
  Award/Insufficient Data — categorical; safe for new vendors)
- **Modeling (Notebook 6):** Linear Regression, Decision Tree, Random
  Forest, XGBoost (regressors), 80/10/10 train/validation/test split,
  cross-validated against Rząd et al.'s reported activity-impact findings
- **Scoring set (not a test set):** unfinished cases in scope — no known
  outcome yet, so the trained model is applied to produce a live
  throughput estimate for in-progress orders, without an accuracy metric

**Part 2 — Vendor Award Prediction:**

- **Target (3-class):** No Award (233) / Bronze (160) / Silver+ = Silver ∪
  Gold (52) — based on `ir_to_clear_days` (matching the Fair Payment Code's
  actual "days from receipt of invoice" definition), 90% SME threshold for
  Silver
- **Sample:** all vendors with ≥30 finished cases producing a valid
  `ir_to_clear_days` — **both 3-way match categories + 2-way match** (not
  Consignment, which structurally cannot produce this metric); **445
  rateable vendors** (1,222 Insufficient Data, 7 Not Applicable, excluded)
- **Unit of analysis:** one row per vendor — aggregated case-level features
  only (% cases per category, median order value, exception-activity
  rate), no historical throughput/outcome data
- **Modeling (Notebook 6):** two variants — **Model A** (existing vendors,
  tested with Logistic Regression and Decision Tree) and **Model B**
  (new/thin-history vendors, using only order-level attributes, tested
  with Logistic Regression, Decision Tree, Random Forest, and XGBoost)
- **Scoring set:** the 1,222 "Insufficient Data" vendors — predict a
  likely tier for vendors without enough history to be reliably rated
  directly

*Methodological note:* Award tiers (Gold/Silver/Bronze) are adapted from
the UK's [Fair Payment Code](https://www.smallbusinesscommissioner.gov.uk/fpc/code-criteria/), with the Silver tier's small-business sub-criterion replaced by a stricter 90%-within-30-days threshold, since [BPI Challenge 2019](https://icpmconference.org/2019/icpm-2019/contests-challenges/bpi-challenge-2019/)
contains no vendor-size field. The 90% small and medium enterprise (SME) threshold is based on [World Bank (2024)](https://www.worldbank.org/ext/en/topic/competitiveness/small-and-medium-enterprises-smes-finance)
and [WEF (2021)](https://www.weforum.org/publications/future-readiness-of-smes-mobilizing-the-sme-sector-to-drive-widespread-sustainability-and-prosperity/), following their finding that SMEs represent around 90 percent of all businesses, adapted here to the vendor dataset. Gold and Silver are merged into a single Silver+ class to resolve a small-sample problem. Real-world day-bucketed payment reporting — e.g. the UK government's
[large business payment practices statistics](https://www.gov.uk/government/statistics/large-businesses-payment-practices-and-performance-statistics-2025/large-businesses-payment-practices-and-performance-statistics-2025-commentary)
— provided precedent for this style of threshold-based reporting in a B2B
payment context. The minimum sample size for a vendor to receive a rating
(n ≥ 30 finished cases) follows the general principle of requiring
statistical validity thresholds before rating a vendor, per
[SPS Commerce's supplier scorecard guidance](https://www.spscommerce.com/community/articles/how-to-build-an-effective-supplier-scorecard)
(their own example uses 50+ units; 30 is used here as a lighter, still
defensible threshold given this dataset's vendor-volume distribution).

## 5.1 Setup

- Imports, pm4py install, load cleaned dataset, format (rebuilds NB4)
- SRM exclusion (rebuilds NB4), with rationale documented inline
- Company/category breakdown + explanation of Part 1's scoping rationale
- Milestone extraction + throughput deltas (rebuilds NB4) — shared
  foundation for both Part 1 and Part 2
- Vendor tier ratings (Gold/Silver/Bronze/No Award/Insufficient
  Data/Not Applicable, min. 30 finished cases, based on `ir_to_clear_days`
  per the Fair Payment Code's actual definition) — shared foundation,
  used as a feature in Part 1 and as the target in Part 2
- Static field constancy check and vendor granularity investigation —
  confirms which fields are safe to use as static features, and documents
  a minor vendor-identifier granularity limitation (`case:Vendor` vs.
  `case:Name`)

In [18]:
# Install dependencies
!pip install pm4py --quiet

In [19]:
# Imports
import pandas as pd
import pm4py
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [20]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
# Load the cleaned dataset from Notebook 1
file_path = "/content/drive/MyDrive/Colab Notebooks/Invoice-to-Payment Process Project/BPI Challenge 2019_Data/BPI_2019_cleaned.parquet"

event_log = pd.read_parquet(file_path)
print(f"Shape: {event_log.shape}")
event_log.head()

Shape: (1595603, 22)


,User,org:resource,concept:name,Cumulative net worth (EUR),time:timestamp,case:Spend area text,case:Company,case:Document Type,case:Sub spend area text,case:Purchasing Document,...,case:Item Type,case:Item Category,case:Spend classification text,case:Source,case:Name,case:GR-Based Inv. Verif.,case:Item,case:concept:name,case:Goods Receipt,event_month
0,batch_00,batch_00,SRM: Created,298.0,2018-01-02 12:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,00001,2000000000_00001,True,2018-01
1,batch_00,batch_00,SRM: Complete,298.0,2018-01-02 13:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,00001,2000000000_00001,True,2018-01
2,batch_00,batch_00,SRM: Awaiting Approval,298.0,2018-01-02 13:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,00001,2000000000_00001,True,2018-01
3,batch_00,batch_00,SRM: Document Completed,298.0,2018-01-02 13:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,00001,2000000000_00001,True,2018-01
4,batch_00,batch_00,SRM: In Transfer to Execution Syst.,298.0,2018-01-02 13:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,00001,2000000000_00001,True,2018-01


In [22]:
# Preserve event_month before pm4py's formatting step potentially alters it
event_month_backup = event_log['event_month'].copy()
event_log_formatted = pm4py.format_dataframe(
    event_log,
    case_id='case:concept:name',
    activity_key='concept:name',
    timestamp_key='time:timestamp'
)
event_log_formatted['event_month'] = event_month_backup

print(f"Events: {event_log_formatted.shape[0]:,}, Columns: {event_log_formatted.shape[1]}")
# Expect: 1,595,603 events, 24 columns

Events: 1,595,603, Columns: 24


In [23]:
# Drop all rows belonging to any case containing an SRM event.
# SRM (Supplier Relationship Management) is a distinct, mostly-automated
# requisition sub-process (1,440 cases, 0.57% of the full dataset) that runs
# before/parallel to the main P2P flow — including it would mix a
# structurally different process into the case-level model. Consistent with
# the SRM exclusion established in Notebook 4 (Section 4.3).

srm_cases = event_log_formatted.loc[
    event_log_formatted['concept:name'].str.startswith('SRM:'), 'case:concept:name'
].unique()
rows_to_drop = event_log_formatted[event_log_formatted['case:concept:name'].isin(srm_cases)].index
event_log_p2p = event_log_formatted.drop(index=rows_to_drop)

print(f"event_log_p2p: {len(event_log_p2p):,} events, {event_log_p2p['case:concept:name'].nunique():,} cases")
# Expect: 1,573,261 events, 250,294 cases

event_log_p2p: 1,573,261 events, 250,294 cases


In [24]:
case_company_category = event_log_p2p.groupby('case:concept:name')[['case:Company', 'case:Item Category']].first()
company_category_mix = case_company_category.groupby('case:Company')['case:Item Category'].value_counts()
print(company_category_mix)

case:Company    case:Item Category            
companyID_0000  3-way match, invoice before GR    220179
                3-way match, invoice after GR      14569
                Consignment                        14498
companyID_0001  3-way match, invoice after GR          2
companyID_0002  3-way match, invoice before GR         2
companyID_0003  2-way match                         1044
Name: count, dtype: int64


In [25]:
# Milestone extraction (4.2.2)
milestones = ['Record Goods Receipt', 'Record Invoice Receipt', 'Clear Invoice']
milestone_events = event_log_p2p[event_log_p2p['concept:name'].isin(milestones)]
first_occurrence = (
    milestone_events
    .sort_values(['case:concept:name', 'time:timestamp'])
    .groupby(['case:concept:name', 'concept:name'])['time:timestamp']
    .first()
    .unstack('concept:name')
)
all_cases = event_log_p2p['case:concept:name'].unique()
first_occurrence_full = first_occurrence.reindex(all_cases)

# Throughput deltas (4.2.3)
throughput = pd.DataFrame(index=first_occurrence_full.index)
throughput['gr_to_ir_days'] = (first_occurrence_full['Record Invoice Receipt'] - first_occurrence_full['Record Goods Receipt']).dt.total_seconds() / 86400
throughput['ir_to_clear_days'] = (first_occurrence_full['Clear Invoice'] - first_occurrence_full['Record Invoice Receipt']).dt.total_seconds() / 86400
throughput['gr_to_clear_days'] = (first_occurrence_full['Clear Invoice'] - first_occurrence_full['Record Goods Receipt']).dt.total_seconds() / 86400

finished_cases = set(event_log_p2p.loc[event_log_p2p['concept:name'] == 'Clear Invoice', 'case:concept:name'])

print(f"throughput: {throughput.shape[0]:,} cases, finished_cases: {len(finished_cases):,}")
# Expect: 250,294 cases, 182,510 finished

throughput: 250,294 cases, finished_cases: 182,510


In [26]:
# Vendor tier ratings, adapted from the UK's Fair Payment Code (Gold/Silver/Bronze),
# using a minimum of 30 finished cases per vendor for statistical reliability

MIN_CASES_FOR_RATING = 30

case_vendor = event_log_p2p.groupby('case:concept:name')['case:Name'].first()
throughput_by_vendor = throughput.join(case_vendor)

# Uses ir_to_clear_days ("days from receipt of invoice"), matching the Fair
# Payment Code's actual definition. This includes 2-way match cases, which
# lack a GR event but still have a valid IR timestamp.
finished_only = throughput_by_vendor[
    throughput_by_vendor.index.isin(finished_cases) & throughput_by_vendor['ir_to_clear_days'].notna()
]

vendor_ratings = finished_only.groupby('case:Name').agg(
    n_cases=('ir_to_clear_days', 'count'),
    pct_within_30=('ir_to_clear_days', lambda x: (x <= 30).mean() * 100),
    pct_within_60=('ir_to_clear_days', lambda x: (x <= 60).mean() * 100),
)

def assign_tier(row):
    if row['n_cases'] < MIN_CASES_FOR_RATING:
        return 'Insufficient Data'
    elif row['pct_within_30'] >= 95:
        return 'Gold'
    elif row['pct_within_60'] >= 95 and row['pct_within_30'] >= 90:
        return 'Silver'
    elif row['pct_within_60'] >= 95:
        return 'Bronze'
    else:
        return 'No Award'

vendor_ratings['tier'] = vendor_ratings.apply(assign_tier, axis=1)

# Vendors with zero valid ir_to_clear_days values fall into two distinct groups:
# - Consignment-only: no invoice activity at all, can never be rated -> "Not Applicable"
# - Has applicable-category cases, but none finished yet -> "Insufficient Data" (n=0),
#   since a rating could still be computed once/if those cases finish

all_vendors = set(event_log_p2p['case:Name'].unique())
rated_vendors = set(vendor_ratings.index)
missing_vendors = all_vendors - rated_vendors

case_vendor_category = event_log_p2p.groupby('case:concept:name')[['case:Name', 'case:Item Category']].first()
missing_vendor_categories = (
    case_vendor_category[case_vendor_category['case:Name'].isin(missing_vendors)]
    .groupby('case:Name')['case:Item Category']
    .apply(lambda x: set(x.unique()))
)

applicable_cats = {'3-way match, invoice before GR', '3-way match, invoice after GR', '2-way match'}
consignment_only = missing_vendor_categories.apply(lambda cats: cats.isdisjoint(applicable_cats))

missing_vendor_tier = pd.DataFrame({
    'tier': consignment_only.map({True: 'Not Applicable', False: 'Insufficient Data'})
})
missing_vendor_tier.index.name = 'case:Name'

vendor_ratings_full = pd.concat([vendor_ratings[['tier']], missing_vendor_tier])

print(f"Total vendors: {len(vendor_ratings_full):,}")
print(vendor_ratings_full['tier'].value_counts())

Total vendors: 1,674
tier
Insufficient Data    1222
No Award              233
Bronze                160
Gold                   39
Silver                 13
Not Applicable          7
Name: count, dtype: int64


**Key insight — Vendor tier ratings:**

Following the UK's [Fair Payment Code](https://www.smallbusinesscommissioner.gov.uk/fpc/code-criteria/),
vendors are rated according to Award tiers (Gold/Silver/Bronze), with the
Silver tier's small-business sub-criterion replaced by a 90%-within-30-days
threshold, as the dataset does not provide information about vendor size.
The results across all 1,674 vendors in the dataset are as follows:

* **Gold:** 39 vendors — pay at least 95% of all invoices within 30 days
* **Silver:** 13 vendors — pay at least 95% of all invoices within 60 days,
  including at least 90% within 30 days
* **Bronze:** 160 vendors — pay at least 95% of all invoices within 60 days
* **No Award:** 233 vendors — above criteria not met
* **Insufficient Data:** 1,222 vendors — fewer than 30 finished cases in an
  applicable category (ranging from 0 to 29), too few for a statistically
  reliable rating
* **Not Applicable:** 7 vendors — every case belongs to the Consignment
  category, which has no invoice activity and can never produce a rating,
  regardless of future data

*Note on methodology:* This tier structure follows real-world payment
reporting conventions — e.g. the UK government's [large business payment practices statistics](https://www.gov.uk/government/statistics/large-businesses-payment-practices-and-performance-statistics-2025/large-businesses-payment-practices-and-performance-statistics-2025-commentary).
The 90% threshold is drawn from general small and medium enterprises (SMEs) population statistics ([World Bank (2024)](https://www.worldbank.org/ext/en/topic/competitiveness/small-and-medium-enterprises-smes-finance);
[WEF (2021)](https://www.weforum.org/publications/future-readiness-of-smes-mobilizing-the-sme-sector-to-drive-widespread-sustainability-and-prosperity/)), not from a direct measurement of this company's own supplier base — no such disclosure exists publicly for this (anonymized) company. The minimum sample size for a rating (n ≥ 30 finished cases) follows the general principle of requiring statistical validity thresholds before rating a vendor, per [SPS Commerce's supplier scorecard guidance](https://www.spscommerce.com/community/articles/how-to-build-an-effective-supplier-scorecard)
(their own example uses 50+ units; 30 is used here as a lighter, still
defensible threshold given this dataset's vendor-volume distribution).

In [27]:
# Sample 1 (Part 1): companyID_0000, both 3-way match categories only
three_way_categories = ['3-way match, invoice before GR', '3-way match, invoice after GR']

sample1_mask = (
    (case_company_category['case:Company'] == 'companyID_0000') &
    (case_company_category['case:Item Category'].isin(three_way_categories))
)
sample1_all = set(case_company_category[sample1_mask].index)
sample1_finished = sample1_all & finished_cases

print(f"Sample 1 (Part 1) — total cases: {len(sample1_all):,}")
print(f"Sample 1 (Part 1) — finished cases: {len(sample1_finished):,}")

# Sample 2 (Part 2): all companies, both 3-way match categories + 2-way match, finished cases only
applicable_categories_part2 = [
    '3-way match, invoice before GR',
    '3-way match, invoice after GR',
    '2-way match'
]
sample2_mask = case_company_category['case:Item Category'].isin(applicable_categories_part2)
sample2_all = set(case_company_category[sample2_mask].index)
sample2_finished = sample2_all & finished_cases

print(f"\nSample 2 (Part 2) — total cases: {len(sample2_all):,}")
print(f"Sample 2 (Part 2) — finished cases: {len(sample2_finished):,}")

Sample 1 (Part 1) — total cases: 234,748
Sample 1 (Part 1) — finished cases: 182,206

Sample 2 (Part 2) — total cases: 235,796
Sample 2 (Part 2) — finished cases: 182,510


**Key insight — Modeling samples:**

Part 1 and Part 2 are based on different, deliberately scoped subsets of the
250,294 non-SRM cases:

- **Sample 1 (Part 1):** `companyID_0000` only, both 3-way match categories
  (invoice before/after GR) — the two categories where a Goods Receipt
  event exists, required for the `gr_to_clear_days` target
- **Sample 2 (Part 2):** all companies, both 3-way match categories plus
  2-way match — 2-way match lacks a GR event but has a valid Invoice
  Receipt, sufficient for `ir_to_clear_days`

*Note:* Consignment cases are excluded from both samples, as these have no
invoice activity and are therefore not valid samples for either target.

In [28]:
# Static field constancy check

exclude_cols = ['time:timestamp', 'concept:name', '@@index', '@@case_index', 'case:concept:name']
all_cols_to_check = [c for c in event_log_p2p.columns if c not in exclude_cols]

constancy_results = []
for col in all_cols_to_check:
    n_distinct_per_case = event_log_p2p.groupby('case:concept:name')[col].nunique()
    n_inconsistent = (n_distinct_per_case > 1).sum()
    constancy_results.append({'column': col, 'cases_with_multiple_values': n_inconsistent})

constancy_df = pd.DataFrame(constancy_results).sort_values('cases_with_multiple_values')
print(constancy_df.to_string(index=False))

                        column  cases_with_multiple_values
          case:Spend area text                           0
      case:Sub spend area text                           0
            case:Document Type                           0
                  case:Company                           0
      case:Purchasing Document                           0
                case:Item Type                           0
                   case:Vendor                           0
case:Purch. Doc. Category name                           0
case:Spend classification text                           0
                   case:Source                           0
                     case:Name                           0
            case:Item Category                           0
     case:GR-Based Inv. Verif.                           0
                     case:Item                           0
            case:Goods Receipt                           0
    Cumulative net worth (EUR)                        46

In [29]:
# Vendor granularity check

vendor_field = event_log_p2p.groupby('case:concept:name')['case:Vendor'].first()
name_field = event_log_p2p.groupby('case:concept:name')['case:Name'].first()
mapping_check = pd.DataFrame({'vendor': vendor_field, 'name': name_field})
vendor_per_name = mapping_check.groupby('name')['vendor'].nunique().sort_values(ascending=False)

print(f"Vendor names with more than one case:Vendor code: {(vendor_per_name > 1).sum()}")
print(vendor_per_name.value_counts().sort_index())

Vendor names with more than one case:Vendor code: 48
vendor
1     1626
2       42
3        2
5        2
10       1
11       1
Name: count, dtype: int64


**Key insight — Static fields and vendor granularity:**

* **Static fields:** All `case:`-prefixed fields are confirmed genuinely
  constant per case, alongside `case:Goods Receipt` and `case:Item` —
  usable as static features with no timing risk. `case:Goods Receipt` is
  excluded, however, as it is fully determined by `case:Item Category`
  and adds no new information.
* **Vendor granularity:** `case:Vendor` provides a more granular identifier
  than `case:Name` for 48 of 1,674 vendors (2.9%), potentially representing
  distinct vendor sub-locations or sub-accounts. All vendor-level
  aggregation in this project (tier ratings, Part 1's `vendor_tier`
  feature, Part 2's scope) uses `case:Name` as one vendor — a
  simplification with no expected material effect.

## 5.2 Part 1 — Case-Level Feature Table

### 5.2.1 Apply Scope

- Restrict to `companyID_0000`, the two 3-way match categories (invoice
  before/after GR), and finished cases only
- Part 1's target requires both a Goods Receipt and a Clear Invoice event,
  and this notebook trains only on cases with a known outcome (see 5.1 for
  the shared `finished_cases` set)

In [30]:
# Part 1 scope: companyID_0000, 3-way match categories only, finished cases only.
# Distinct from Part 2, which also includes 2-way match across all companies.

case_company_category = event_log_p2p.groupby('case:concept:name')[['case:Company', 'case:Item Category']].first()

three_way_categories = ['3-way match, invoice before GR', '3-way match, invoice after GR']
in_scope_mask = (
    (case_company_category['case:Company'] == 'companyID_0000') &
    (case_company_category['case:Item Category'].isin(three_way_categories))
)
sample1_all = set(case_company_category[in_scope_mask].index)
sample1_finished = sample1_all & finished_cases

print(f"Sample 1 (Part 1) — total cases: {len(sample1_all):,}")
print(f"Sample 1 (Part 1) — finished cases: {len(sample1_finished):,}")
# Expect: 234,748 total, 182,206 finished

Sample 1 (Part 1) — total cases: 234,748
Sample 1 (Part 1) — finished cases: 182,206


### 5.2.2 Feature Construction

Building Part 1's feature table in three stages:
1. Static features first (no temporal filtering needed)
2. Exception-activity flags (filtered against each case's own GR
   timestamp)
3. Vendor-tier join

**Stage 1 — Static features:** item category, spend area, document type,
and order value. These are fixed at PO creation, well before Goods Receipt,
so no leakage risk applies here.

In [31]:
# Restrict event log to Sample 1's scope (companyID_0000, both 3-way match
# categories, finished cases only) before extracting features — needed by
# the static-features cell below
part1_events = event_log_p2p[event_log_p2p['case:concept:name'].isin(sample1_finished)]

print(f"part1_events shape: {part1_events.shape}")
print(f"Unique cases: {part1_events['case:concept:name'].nunique():,}")

part1_events shape: (1179508, 24)
Unique cases: 182,206


In [32]:
static_fields = [
    'case:Item Category', 'case:Spend area text', 'case:Sub spend area text',
    'case:Document Type', 'case:Purchasing Document', 'case:Item Type',
    'case:Purch. Doc. Category name', 'case:Spend classification text',
    'case:Source', 'case:GR-Based Inv. Verif.', 'case:Item',
    'Cumulative net worth (EUR)'
]

part1_features = part1_events.groupby('case:concept:name').first()[static_fields]

part1_features = part1_features.rename(columns={
    'case:Item Category': 'item_category',
    'case:Spend area text': 'spend_area',
    'case:Sub spend area text': 'sub_spend_area',
    'case:Document Type': 'document_type',
    'case:Purchasing Document': 'purchasing_document',
    'case:Item Type': 'item_type',
    'case:Purch. Doc. Category name': 'purch_doc_category',
    'case:Spend classification text': 'spend_classification',
    'case:Source': 'source',
    'case:GR-Based Inv. Verif.': 'gr_based_inv_verif',
    'case:Item': 'item',
    'Cumulative net worth (EUR)': 'order_value'
})

print(f"Static feature table shape: {part1_features.shape}")
# Expect: (182206, 12)
print(f"\nMissing values per column:")
print(part1_features.isna().sum())

Static feature table shape: (182206, 12)

Missing values per column:
item_category           0
spend_area              0
sub_spend_area          0
document_type           0
purchasing_document     0
item_type               0
purch_doc_category      0
spend_classification    0
source                  0
gr_based_inv_verif      0
item                    0
order_value             0
dtype: int64


In [33]:
# Cardinality check for purchasing_document and item

print(f"case:Purchasing Document — distinct values: {part1_features['purchasing_document'].nunique():,} (of {len(part1_features):,} cases)")
print(f"case:Item — distinct values: {part1_features['item'].nunique():,} (of {len(part1_features):,} cases)")

case:Purchasing Document — distinct values: 52,197 (of 182,206 cases)
case:Item — distinct values: 311 (of 182,206 cases)


In [34]:
# Investigate 311 distinct values

print(part1_features['item'].value_counts().head(10))

item
00010    47011
00020    21184
00030    14437
00040    11078
00050     9003
00060     7552
00070     6503
00080     5709
00090     5003
00100     4434
Name: count, dtype: int64


In [35]:
part1_features = part1_features.rename(columns={'item': 'item_line_position'})

*Note on feature exclusions:* `purchasing_document` (52,197 distinct values,
28.6% cardinality) is excluded from the model feature matrix — too
high-cardinality to generalize from, functioning as a near-identifier. It is
retained in the exported table for traceability and potential future
grouping analysis. `item_line_position` (311 distinct values) is retained
as a genuine categorical feature, confirmed to represent a line item's
position within its purchase order (SAP convention, increments of 10)
rather than a product/material identifier.

**Stage 2 — Exception-activity flags:** binary indicators for whether each
"Undesired Activity" (Table 2 taxonomy of [Rząd, Wojnecka, Rutkowski & Guliński (2019)](https://icpmconference.org/2019/wp-content/uploads/sites/6/2019/07/BPI-Challenge-Submission-2.pdf)) occurred
**at or before** the case's own Goods Receipt timestamp. This timestamp
filter is essential — an activity occurring after GR would not have been
knowable at the model's chosen prediction point, and including it would
leak information about the case's future into a feature meant to predict
that future (see 5.2.1 discussion on prediction point).

In [36]:
# Rząd et al. (2019) "Undesired Activities" taxonomy (Table 2)
undesired_activities = [
    'Block Purchase Order Item', 'Cancel Goods Receipt', 'Cancel Invoice Receipt',
    'Change Approval for Purchase Order', 'Change Currency', 'Change payment term',
    'Change Price', 'Change Quantity', 'Change Storage Location',
    'Delete Purchase Order Item', 'Record Subsequent Invoice', 'Update Order Confirmation'
]

# Get each case's own GR timestamp (first occurrence), to filter against
case_gr_time = part1_events[part1_events['concept:name'] == 'Record Goods Receipt'] \
    .groupby('case:concept:name')['time:timestamp'].first()

# Filter to only undesired-activity events occurring AT OR BEFORE that case's GR timestamp
undesired_events = part1_events[part1_events['concept:name'].isin(undesired_activities)].copy()
undesired_events['case_gr_time'] = undesired_events['case:concept:name'].map(case_gr_time)
undesired_events_pre_gr = undesired_events[undesired_events['time:timestamp'] <= undesired_events['case_gr_time']]

# Build binary flags: one column per activity, 1 if it occurred pre-GR for that case, else 0
exception_flags = (
    undesired_events_pre_gr
    .groupby(['case:concept:name', 'concept:name'])
    .size()
    .unstack(fill_value=0)
    .clip(upper=1)  # convert counts to binary
)

# Reindex to all of Sample 1's finished cases, filling 0 for cases with no pre-GR exceptions at all
exception_flags = exception_flags.reindex(sample1_finished, fill_value=0)
exception_flags.columns = [f'flag_{c.lower().replace(" ", "_")}' for c in exception_flags.columns]

print(f"Exception flags table shape: {exception_flags.shape}")
print(f"\nHow often each flag fires (pre-GR only):")
print(exception_flags.sum().sort_values(ascending=False))

Exception flags table shape: (182206, 11)

How often each flag fires (pre-GR only):
flag_change_quantity                       11332
flag_change_price                           6095
flag_change_approval_for_purchase_order     1633
flag_change_storage_location                 229
flag_delete_purchase_order_item              226
flag_update_order_confirmation               176
flag_block_purchase_order_item               131
flag_cancel_invoice_receipt                   80
flag_cancel_goods_receipt                     59
flag_change_currency                          26
flag_change_payment_term                       4
dtype: int64


In [37]:
# Compare pre-GR-filtered rates against unfiltered full-case rates,
# to show concretely what the prediction-point discipline changes

all_undesired_events = part1_events[part1_events['concept:name'].isin(undesired_activities)]
full_case_flags = (
    all_undesired_events
    .groupby(['case:concept:name', 'concept:name'])
    .size()
    .unstack(fill_value=0)
    .clip(upper=1)
    .reindex(sample1_finished, fill_value=0)
)
full_case_flags.columns = [f'flag_{c.lower().replace(" ", "_")}' for c in full_case_flags.columns]
full_case_totals = full_case_flags.reindex(columns=exception_flags.columns, fill_value=0).sum()

comparison = pd.DataFrame({
    'pre_GR_only': exception_flags.sum(),
    'full_case_unfiltered': full_case_totals
})
comparison['pct_removed_by_filter'] = (
    (comparison['full_case_unfiltered'] - comparison['pre_GR_only']) / comparison['full_case_unfiltered'] * 100
).round(1)
comparison.sort_values('full_case_unfiltered', ascending=False)

,pre_GR_only,full_case_unfiltered,pct_removed_by_filter
flag_change_quantity,11332,12748,11.1
flag_change_price,6095,8977,32.1
flag_cancel_invoice_receipt,80,5885,98.6
flag_change_approval_for_purchase_order,1633,1644,0.7
flag_cancel_goods_receipt,59,1206,95.1
flag_delete_purchase_order_item,226,274,17.5
flag_change_storage_location,229,247,7.3
flag_update_order_confirmation,176,187,5.9
flag_block_purchase_order_item,131,163,19.6
flag_change_currency,26,26,0.0


In [38]:
# Check: are the 59 "pre-GR" Cancel Goods Receipt events genuinely before GR,
# or same-timestamp batch artifacts?

cancel_gr_events = part1_events[part1_events['concept:name'] == 'Cancel Goods Receipt'].copy()
cancel_gr_events['case_gr_time'] = cancel_gr_events['case:concept:name'].map(case_gr_time)
pre_gr_cancel_gr = cancel_gr_events[cancel_gr_events['time:timestamp'] <= cancel_gr_events['case_gr_time']]

same_timestamp = (pre_gr_cancel_gr['time:timestamp'] == pre_gr_cancel_gr['case_gr_time']).sum()
print(f"Pre-GR Cancel Goods Receipt events: {len(pre_gr_cancel_gr)}")
print(f"Of those, exactly simultaneous with GR: {same_timestamp}")

Pre-GR Cancel Goods Receipt events: 59
Of those, exactly simultaneous with GR: 59


In [39]:
# Rebuild exception flags with strict < (not <=) — same-timestamp events
# shouldn't count as "known before GR", confirmed by the Cancel Goods Receipt check

undesired_events_pre_gr = undesired_events[undesired_events['time:timestamp'] < undesired_events['case_gr_time']]

exception_flags = (
    undesired_events_pre_gr
    .groupby(['case:concept:name', 'concept:name'])
    .size()
    .unstack(fill_value=0)
    .clip(upper=1)
)
exception_flags = exception_flags.reindex(sample1_finished, fill_value=0)
exception_flags.columns = [f'flag_{c.lower().replace(" ", "_")}' for c in exception_flags.columns]

print(f"Exception flags table shape: {exception_flags.shape}")
print(exception_flags.sum().sort_values(ascending=False))

Exception flags table shape: (182206, 10)
flag_change_quantity                       11218
flag_change_price                           6091
flag_change_approval_for_purchase_order     1633
flag_change_storage_location                 229
flag_delete_purchase_order_item              226
flag_update_order_confirmation               176
flag_block_purchase_order_item               131
flag_cancel_invoice_receipt                   80
flag_change_currency                          26
flag_change_payment_term                       4
dtype: int64


In [40]:
# Check same-timestamp overlap across ALL undesired activities, not just Cancel Goods Receipt

undesired_events_check = undesired_events.copy()
undesired_events_check['is_same_timestamp'] = undesired_events_check['time:timestamp'] == undesired_events_check['case_gr_time']
print(undesired_events_check.groupby('concept:name')['is_same_timestamp'].sum().sort_values(ascending=False))

concept:name
Change Quantity                       141
Cancel Goods Receipt                   59
Change Price                            4
Block Purchase Order Item               0
Change Approval for Purchase Order      0
Cancel Invoice Receipt                  0
Change Currency                         0
Change Storage Location                 0
Change payment term                     0
Delete Purchase Order Item              0
Record Subsequent Invoice               0
Update Order Confirmation               0
Name: is_same_timestamp, dtype: int64


*Note on timestamp boundary:* Exception-activity flags use a strict "before
GR" rule (`<`, not `<=`). Checking confirmed 200 events across three
activities (Cancel Goods Receipt: 59, Change Quantity: 141, Change Price: 4)
shared an identical timestamp with their case's GR event — likely batch
processing artifacts (see Notebook 1's documented findings on `NONE`-resource,
identical-timestamp events). These are excluded, since a same-timestamp
event cannot be considered genuinely knowable *before* GR at the model's
chosen prediction point.

In [41]:
# Stage 2: join the already-validated exception-activity flags (with the
# strict pre-GR timestamp filter) onto the expanded static feature table
part1_features = part1_features.join(exception_flags)

print(f"Shape after joining exception flags: {part1_features.shape}")
print(f"Flag columns present: {[c for c in part1_features.columns if c.startswith('flag_')]}")
# Expect: (182206, 22) — 12 static + 10 flags

Shape after joining exception flags: (182206, 22)
Flag columns present: ['flag_block_purchase_order_item', 'flag_cancel_invoice_receipt', 'flag_change_approval_for_purchase_order', 'flag_change_currency', 'flag_change_price', 'flag_change_quantity', 'flag_change_storage_location', 'flag_change_payment_term', 'flag_delete_purchase_order_item', 'flag_update_order_confirmation']


**Stage 3 — Vendor tier:** join each case's vendor's rating tier
(Gold/Silver/Bronze/No Award/Insufficient Data/Not Applicable) from the
shared `vendor_ratings_full` table (5.1) as a categorical feature. Since
this is built entirely from a vendor's *other* completed cases — never a predicted one — it carries no leakage risk regardless of the chosen prediction
point.

In [42]:
# Join vendor identity, then vendor tier — both at the case level

case_vendor_map = part1_events.groupby('case:concept:name')['case:Name'].first()

part1_features['vendor'] = case_vendor_map
part1_features = part1_features.join(vendor_ratings_full[['tier']].rename(columns={'tier': 'vendor_tier'}), on='vendor')

print(f"part1_features shape: {part1_features.shape}")
print(f"\nMissing vendor_tier values: {part1_features['vendor_tier'].isna().sum()}")
print(f"\nvendor_tier distribution within Sample 1:")
print(part1_features['vendor_tier'].value_counts())

part1_features shape: (182206, 24)

Missing vendor_tier values: 0

vendor_tier distribution within Sample 1:
vendor_tier
No Award             103546
Bronze                57644
Gold                   9779
Insufficient Data      6751
Silver                 4486
Name: count, dtype: int64


*Note:* "Not Applicable" does not appear in this distribution — by
definition, every vendor with that tier has zero cases in either 3-way
match category (all-Consignment only), so none can appear within Sample 1's
scope. Separately, vendor tiers are weighted by *case volume* here, not vendor count. As a result, no Award vendors alone account for 56.8% of Sample 1's cases (103,546 of 182,206), despite representing only 233 of the 1,674 total vendors.

In [43]:
# Expansion static feature table

part1_features['vendor'] = case_vendor_map
part1_features = part1_features.drop(columns=['vendor_tier'], errors='ignore')
part1_features = part1_features.join(vendor_ratings_full[['tier']].rename(columns={'tier': 'vendor_tier'}), on='vendor')

print(f"Shape after vendor join: {part1_features.shape}")
print(f"Missing vendor_tier values: {part1_features['vendor_tier'].isna().sum()}")

Shape after vendor join: (182206, 24)
Missing vendor_tier values: 0


### 5.2.3 Target Variable

- Extract `gr_to_clear_days` (continuous) from the shared `throughput`
  table (5.1), for the same 182,206 cases in Sample 1

In [44]:
part1_features['target_gr_to_clear_days'] = throughput.loc[part1_features.index, 'gr_to_clear_days']

print(f"Missing target values: {part1_features['target_gr_to_clear_days'].isna().sum()}")
print(f"\nTarget distribution:")
print(part1_features['target_gr_to_clear_days'].describe())

Missing target values: 566

Target distribution:
count    181640.000000
mean         65.887995
std          30.524482
min        -161.959028
25%          44.841667
50%          63.049306
75%          85.899479
max         345.220139
Name: target_gr_to_clear_days, dtype: float64


In [45]:
# Investigate the 566 missing targets — hypothesis: these cases have Clear Invoice
# (so they count as "finished") but are missing a Goods Receipt event

missing_target_cases = part1_features[part1_features['target_gr_to_clear_days'].isna()].index

check = first_occurrence_full.loc[missing_target_cases]
print(f"Missing Goods Receipt: {check['Record Goods Receipt'].isna().sum()}")
print(f"Missing Clear Invoice: {check['Clear Invoice'].isna().sum()}")

Missing Goods Receipt: 566
Missing Clear Invoice: 0


In [46]:
# Investigate scale before dropping, per project convention

n_negative = (part1_features['target_gr_to_clear_days'] < 0).sum()
print(f"Negative target values: {n_negative} ({n_negative / len(part1_features) * 100:.2f}%)")

Negative target values: 88 (0.05%)


In [47]:
# Drop 566 missing-target rows, then negative values (invalid per process logic)

part1_features_clean = part1_features.dropna(subset=['target_gr_to_clear_days'])
part1_features_clean = part1_features_clean[part1_features_clean['target_gr_to_clear_days'] >= 0]

print(f"Final shape: {part1_features_clean.shape}")
print(part1_features_clean['target_gr_to_clear_days'].describe())

Final shape: (181552, 25)
count    181552.000000
mean         65.933924
std          30.452000
min           0.091667
25%          44.857639
50%          63.055556
75%          85.910590
max         345.220139
Name: target_gr_to_clear_days, dtype: float64


In [48]:
# Expansion static feature table

part1_features['target_gr_to_clear_days'] = throughput.loc[part1_features.index, 'gr_to_clear_days']

part1_features_clean = part1_features.dropna(subset=['target_gr_to_clear_days'])
part1_features_clean = part1_features_clean[part1_features_clean['target_gr_to_clear_days'] >= 0]

print(f"Final shape: {part1_features_clean.shape}")
# Expect: (181552, 25)

Final shape: (181552, 25)


*Note:* target cleaning is detailed in 5.2.4 below, alongside the export's
final column summary.

Final training population: **181,552 cases** (99.64% of Sample 1's original
182,206).

### 5.2.4 Export

- Save Part 1's complete, cleaned feature table (12 static fields, 10
  exception-activity flags, vendor + vendor_tier, target) for use in
  Notebook 6

In [49]:
export_path = "/content/drive/MyDrive/Colab Notebooks/Invoice-to-Payment Process Project/BPI Challenge 2019_Data/part1_features.parquet"
part1_features_clean.to_parquet(export_path)

print(f"Exported: {part1_features_clean.shape}")
print(f"Columns: {list(part1_features_clean.columns)}")

Exported: (181552, 25)
Columns: ['item_category', 'spend_area', 'sub_spend_area', 'document_type', 'purchasing_document', 'item_type', 'purch_doc_category', 'spend_classification', 'source', 'gr_based_inv_verif', 'item_line_position', 'order_value', 'flag_block_purchase_order_item', 'flag_cancel_invoice_receipt', 'flag_change_approval_for_purchase_order', 'flag_change_currency', 'flag_change_price', 'flag_change_quantity', 'flag_change_storage_location', 'flag_change_payment_term', 'flag_delete_purchase_order_item', 'flag_update_order_confirmation', 'vendor', 'vendor_tier', 'target_gr_to_clear_days']


*Note on feature exclusions:* `purchasing_document` (52,197 distinct
values, 28.6% cardinality) is retained in this exported table for
traceability, but should be excluded from the model's feature matrix in
Notebook 6 — too high-cardinality to generalize from. It is also worth
using as a grouping key when splitting train/validation/test, since cases
sharing a purchasing document likely share underlying business context and
should not be split across sets. `item_line_position` is a genuine
categorical feature (SAP line-numbering convention, increments of 10) — not
a product/material identifier, despite the generic column name it was
derived from.

*Note on target cleaning:* 566 cases (0.31%) were dropped for missing a
Goods Receipt event despite reaching Clear Invoice. A further 88 cases
(0.05%) were dropped for a negative `gr_to_clear_days` — a logical
impossibility, consistent with the first-occurrence-matching artifacts
documented in 4.2.3/4.5.1. Extreme positive durations were retained, since
unusually slow — but logically valid — cases are exactly what the
delay-prediction model is meant to learn to recognize; right-skew is
addressed via a log-transform for Linear Regression specifically in
Notebook 6.

Final training population: **181,552 cases** (99.64% of Sample 1's
original 182,206), with **25 columns** (12 static fields, 10
exception-activity flags, vendor identity, vendor tier, and target).

## 5.3 Part 2 — Vendor-Level Feature Table


### 5.3.1 Apply Scope

- Restrict to rateable vendors only — those with a Gold, Silver, Bronze, or
  No Award tier (excludes "Insufficient Data" and "Not Applicable", which
  form Part 2's later scoring set)
- Unlike Part 1, not restricted to `companyID_0000` — draws on Sample 2's
  broader scope (both 3-way match categories + 2-way match, all companies)
- This scope (`rateable_vendors`, `part2_cases`, `part2_events`) feeds
  **both** Model A and Model B below, since Model B uses the same
  underlying transaction population, just at finer granularity

In [50]:
# Rateable vendors: Gold, Silver, Bronze, No Award only
rateable_tiers = ['Gold', 'Silver', 'Bronze', 'No Award']
rateable_vendors = vendor_ratings_full[vendor_ratings_full['tier'].isin(rateable_tiers)].index

print(f"Rateable vendors: {len(rateable_vendors):,}")
print(vendor_ratings_full.loc[rateable_vendors, 'tier'].value_counts())
# Expect: 445 total — No Award 233, Bronze 160, Gold 39, Silver 13

# Cases belonging to rateable vendors, within Sample 2's applicable categories
case_vendor_full = event_log_p2p.groupby('case:concept:name')['case:Name'].first()
part2_cases = set(case_vendor_full[case_vendor_full.isin(rateable_vendors)].index) & sample2_all

print(f"\nCases feeding Part 2's vendor-level aggregation: {len(part2_cases):,}")
# Expect: 221,242

Rateable vendors: 445
tier
No Award    233
Bronze      160
Gold         39
Silver       13
Name: count, dtype: int64

Cases feeding Part 2's vendor-level aggregation: 221,242


### 5.3.2 Vendor-Level Feature Aggregation

- One row per vendor (445 rateable vendors), built from their case history
  within Part 2's scope (both 3-way match categories + 2-way match)
- Category mix: % of each vendor's cases falling into each `item_category`
- Order value: median and count of cases
- Exception-activity rate: % of a vendor's cases containing each
  Undesired Activity (Rząd et al., 2019) — computed over the full case,
  not filtered to a prediction point, since this describes a general
  behavioral tendency rather than predicting any single case's own outcome
- Document type and spend classification mix: % of each vendor's cases
  falling into each `document_type` and `spend_classification` value
- Explicitly excludes any feature derived from `ir_to_clear_days` or
  `gr_to_clear_days` — the tier itself is the target; using outcome-derived
  features here would be circular

In [51]:
part2_events = event_log_p2p[event_log_p2p['case:concept:name'].isin(part2_cases)]

# Category mix: % of each vendor's cases in each item_category
case_vendor_category_full = part2_events.groupby('case:concept:name')[['case:Name', 'case:Item Category']].first()
category_mix = (
    pd.crosstab(case_vendor_category_full['case:Name'], case_vendor_category_full['case:Item Category'], normalize='index') * 100
)
category_mix.columns = [f'pct_{c.lower().replace(" ", "_").replace(",", "")}' for c in category_mix.columns]

# Order value: median + case count per vendor
case_vendor_value = part2_events.groupby('case:concept:name')[['case:Name', 'Cumulative net worth (EUR)']].first()
order_value_agg = case_vendor_value.groupby('case:Name')['Cumulative net worth (EUR)'].agg(
    median_order_value='median',
    n_cases='count'
)

part2_features = category_mix.join(order_value_agg)

print(f"Part 2 feature table shape (before exception rates): {part2_features.shape}")
print(part2_features.head())

Part 2 feature table shape (before exception rates): (445, 5)
             pct_2-way_match  pct_3-way_match_invoice_after_gr  \
case:Name                                                        
vendor_0032              0.0                          0.000000   
vendor_0036              0.0                         22.142857   
vendor_0042              0.0                          2.105263   
vendor_0056              0.0                          0.000000   
vendor_0060              0.0                          0.000000   

             pct_3-way_match_invoice_before_gr  median_order_value  n_cases  
case:Name                                                                    
vendor_0032                         100.000000              1794.0       87  
vendor_0036                          77.857143              3550.0      140  
vendor_0042                          97.894737              2323.0       95  
vendor_0056                         100.000000              4885.0      175  
vendor_

In [52]:
# Compare this table's n_cases against the finished-only count used for tier rating
comparison = pd.DataFrame({
    'n_cases_all_status': part2_features['n_cases'],
    'n_cases_finished_only': vendor_ratings.loc[part2_features.index, 'n_cases']
})
comparison['diff'] = comparison['n_cases_all_status'] - comparison['n_cases_finished_only']
print(comparison['diff'].describe())

count     445.000000
mean      103.662921
std       341.697819
min         0.000000
25%         9.000000
50%        21.000000
75%        70.000000
max      3873.000000
Name: diff, dtype: float64


*Note:* `n_cases` in this table counts all cases in Part 2's scope
regardless of finish status, while the vendor's tier rating (5.1) uses
finished cases only. This is intentional — category mix and order value
are not outcome-dependent, so including open cases here provides a fuller,
still leakage-free behavioral profile. The gap between the two counts
(median 21, max 3,873) reflects each vendor's current share of in-progress
orders.

In [53]:
undesired_events_part2 = part2_events[part2_events['concept:name'].isin(undesired_activities)]

case_vendor_map_part2 = part2_events.groupby('case:concept:name')['case:Name'].first()

case_has_activity = (
    undesired_events_part2
    .groupby(['case:concept:name', 'concept:name'])
    .size()
    .unstack(fill_value=0)
    .clip(upper=1)
)
case_has_activity = case_has_activity.reindex(part2_cases, fill_value=0)
case_has_activity['case:Name'] = case_vendor_map_part2

exception_rates = case_has_activity.groupby('case:Name').mean(numeric_only=True) * 100
exception_rates.columns = [f'rate_{c.lower().replace(" ", "_")}' for c in exception_rates.columns]

part2_features = part2_features.join(exception_rates)

print(f"Part 2 feature table shape: {part2_features.shape}")
print(f"\nException rate columns:")
print(exception_rates.mean().sort_values(ascending=False))

Part 2 feature table shape: (445, 17)

Exception rate columns:
rate_change_quantity                       8.892799
rate_change_price                          6.823814
rate_cancel_invoice_receipt                2.977430
rate_delete_purchase_order_item            2.902328
rate_cancel_goods_receipt                  1.252748
rate_change_approval_for_purchase_order    0.725838
rate_block_purchase_order_item             0.323096
rate_change_storage_location               0.165828
rate_update_order_confirmation             0.143047
rate_record_subsequent_invoice             0.082435
rate_change_currency                       0.012002
rate_change_payment_term                   0.002968
dtype: float64


*Note:* `rate_record_subsequent_invoice` shows a small nonzero rate (0.08%)
here, despite this same activity being entirely absent from Part 1's
pre-GR-filtered flags. This is expected, not a discrepancy — Part 1 asks
whether an activity occurred *before that specific case's own GR* (a
prediction-point question), while Part 2 asks whether it occurs *anywhere*
in a vendor's history (a behavioral-tendency question). The two parts are
deliberately answering different questions about the same underlying data.

In [54]:
# Check remaining static fields (document_type, spend_classification, source,
# purch_doc_category) within Part 2's own scope, before deciding whether to add them

part2_static_check = part2_events.groupby('case:concept:name').first()

for col, label in [
    ('case:Document Type', 'document_type'),
    ('case:Spend classification text', 'spend_classification'),
    ('case:Source', 'source'),
    ('case:Purch. Doc. Category name', 'purch_doc_category')
]:
    n_distinct = part2_static_check[col].nunique()
    print(f"{label}: {n_distinct} distinct values (Part 2 scope)")
    if n_distinct <= 5:
        print(f"  {part2_static_check[col].value_counts().to_dict()}")

document_type: 2 distinct values (Part 2 scope)
  {'Standard PO': 221003, 'Framework order': 239}
spend_classification: 4 distinct values (Part 2 scope)
  {'PR': 144266, 'NPR': 71923, '': 2927, 'OTHER': 2126}
source: 1 distinct values (Part 2 scope)
  {'sourceSystemID_0000': 221242}
purch_doc_category: 1 distinct values (Part 2 scope)
  {'Purchase order': 221242}


In [55]:
# source and purch_doc_category are constant (no predictive value) - excluded.
# document_type and spend_classification genuinely vary - add as vendor-level
# category-mix features. Empty string in spend_classification relabeled "Missing".

part2_static_check['case:Spend classification text'] = part2_static_check['case:Spend classification text'].replace('', 'Missing')

doctype_mix = pd.crosstab(case_vendor_category_full['case:Name'], part2_static_check['case:Document Type'], normalize='index') * 100
doctype_mix.columns = [f'pct_doctype_{c.lower().replace(" ", "_")}' for c in doctype_mix.columns]

spendclass_mix = pd.crosstab(case_vendor_category_full['case:Name'], part2_static_check['case:Spend classification text'], normalize='index') * 100
spendclass_mix.columns = [f'pct_spendclass_{c.lower().replace(" ", "_")}' for c in spendclass_mix.columns]

# Drop these columns first if already present, so this cell is safely re-runnable
part2_features = part2_features.drop(columns=list(doctype_mix.columns) + list(spendclass_mix.columns), errors='ignore')
part2_features = part2_features.join(doctype_mix).join(spendclass_mix)

print(f"Part 2 feature table shape: {part2_features.shape}")
# Expect: (445, 23)

Part 2 feature table shape: (445, 23)


In [56]:
# Confirm no unexpected missing values before finalizing

print(f"Total missing values across all columns: {part2_features.isna().sum().sum()}")
print(part2_features.isna().sum()[part2_features.isna().sum() > 0])

Total missing values across all columns: 0
Series([], dtype: int64)


In [57]:
# Join the target: vendor tier, reduced to the agreed 3-class structure
# (No Award / Bronze / Silver+), merging Silver and Gold to resolve the
# small-sample problem established earlier in this project

tier_target = vendor_ratings_full.loc[part2_features.index, 'tier']
target_3class = tier_target.replace({'Silver': 'Silver+', 'Gold': 'Silver+'})

part2_features['target_award_tier'] = target_3class

print(part2_features['target_award_tier'].value_counts())
# Expect: No Award 233, Bronze 160, Silver+ 52

target_award_tier
No Award    233
Bronze      160
Silver+      52
Name: count, dtype: int64


**Target: 3-class Award tier.** Part 2's raw tier distribution (5.1) has
six categories, two of which are out of scope for prediction entirely
("Insufficient Data" and "Not Applicable" form the later scoring set, not
training data). Of the four remaining tiers, Gold and Silver are both too
small to model reliably on their own — under the `gr_to_clear_days`-based
tiers used earlier in development, Gold sat at just 13 vendors; after
correcting the tier basis to `ir_to_clear_days` (5.1), Silver became the
thin one instead, at 13. Whichever tier ends up smaller, the underlying
issue is structural: stricter tiers will always attract fewer vendors, by
definition of being stricter.

Silver and Gold are therefore merged into a single **Silver+** class,
bringing that group to 52 vendors — a more statistically stable sample
size for macro-averaged evaluation than either tier alone — while No
Award (233) and Bronze (160) remain untouched. This gives a 3-class target
with a roughly 4.5:3:1 balance, still imbalanced but workable with
`class_weight='balanced'` and macro-averaged evaluation metrics in
Notebook 6.

### 5.3.3 Export — Model A

- Save Part 2's complete vendor-level feature table (category mix, order
  value, exception-activity rates, document type/spend classification mix,
  3-class Award tier target) for use in Notebook 6

In [58]:
export_path = "/content/drive/MyDrive/Colab Notebooks/Invoice-to-Payment Process Project/BPI Challenge 2019_Data/part2a_features.parquet"
part2_features.to_parquet(export_path)

print(f"Exported: {part2_features.shape}")
print(f"Columns: {list(part2_features.columns)}")

Exported: (445, 24)
Columns: ['pct_2-way_match', 'pct_3-way_match_invoice_after_gr', 'pct_3-way_match_invoice_before_gr', 'median_order_value', 'n_cases', 'rate_block_purchase_order_item', 'rate_cancel_goods_receipt', 'rate_cancel_invoice_receipt', 'rate_change_approval_for_purchase_order', 'rate_change_currency', 'rate_change_price', 'rate_change_quantity', 'rate_change_storage_location', 'rate_change_payment_term', 'rate_delete_purchase_order_item', 'rate_record_subsequent_invoice', 'rate_update_order_confirmation', 'pct_doctype_framework_order', 'pct_doctype_standard_po', 'pct_spendclass_missing', 'pct_spendclass_npr', 'pct_spendclass_other', 'pct_spendclass_pr', 'target_award_tier']


Final export: **445 vendors** × **24 columns** (vendor-level aggregates:
category mix, order value, exception rates, document type/spend
classification mix, plus the 3-class Award tier target) — no outcome-derived
features, keeping the table applicable to vendors with thin-but-nonzero
history.

### 5.3.4 Model B — New-Vendor Feature Table

- One row per transaction (not per vendor), using only order-level static
  fields knowable before any vendor-specific history exists
- `gr_based_inv_verif` excluded — confirmed deterministically mapped to
  `item_category` (same redundancy as `case:Goods Receipt`), adds no new
  information
- Label: the transaction's vendor's Award tier (3-class, matching Model A)
- Train/test split must be grouped by vendor (`GroupShuffleSplit`), not by
  row — otherwise cases from the same vendor could appear in both sets,
  letting the model implicitly learn vendor-specific patterns even without
  vendor identity as a feature (documented here for Notebook 6)

In [59]:
model_b_static_fields = [
    'case:Item Category', 'case:Spend area text', 'case:Sub spend area text',
    'case:Spend classification text', 'case:Document Type', 'case:Item Type',
    'case:Item', 'Cumulative net worth (EUR)'
]

# Restrict to the same case population as Model A: rateable vendors only,
# within Part 2's applicable categories
model_b_events = event_log_p2p[event_log_p2p['case:concept:name'].isin(part2_cases)]

model_b_features = model_b_events.groupby('case:concept:name').first()[model_b_static_fields]

model_b_features = model_b_features.rename(columns={
    'case:Item Category': 'item_category',
    'case:Spend area text': 'spend_area',
    'case:Sub spend area text': 'sub_spend_area',
    'case:Spend classification text': 'spend_classification',
    'case:Document Type': 'document_type',
    'case:Item Type': 'item_type',
    'case:Item': 'item_line_position',
    'Cumulative net worth (EUR)': 'order_value'
})

print(f"Model B feature table shape: {model_b_features.shape}")
# Expect: (221242, 8)

Model B feature table shape: (221242, 8)


In [60]:
# Attach each transaction's vendor and that vendor's Award tier (3-class, matching Model A)
case_vendor_map_part2 = event_log_p2p.groupby('case:concept:name')['case:Name'].first()
model_b_features['vendor'] = case_vendor_map_part2

tier_lookup = vendor_ratings_full.loc[model_b_features['vendor'], 'tier'].replace({'Silver': 'Silver+', 'Gold': 'Silver+'})
model_b_features['target_award_tier'] = tier_lookup.values

print(f"Shape after label join: {model_b_features.shape}")
print(f"\nMissing target values: {model_b_features['target_award_tier'].isna().sum()}")
print(f"\nTransaction-level target distribution:")
print(model_b_features['target_award_tier'].value_counts())
print(f"\n(Percentage view, for comparison against Model A's vendor-level 233/160/52 split)")
print((model_b_features['target_award_tier'].value_counts(normalize=True) * 100).round(1))

Shape after label join: (221242, 10)

Missing target values: 0

Transaction-level target distribution:
target_award_tier
No Award    139658
Bronze       65881
Silver+      15703
Name: count, dtype: int64

(Percentage view, for comparison against Model A's vendor-level 233/160/52 split)
target_award_tier
No Award    63.1
Bronze      29.8
Silver+      7.1
Name: proportion, dtype: float64


In [61]:
print(f"Unique vendors in Model B: {model_b_features['vendor'].nunique()}")
# Expect: 445 — matches Model A's rateable vendor count exactly

Unique vendors in Model B: 445


*Note for Notebook 6:* the `vendor` column must be used as the grouping key
for `GroupShuffleSplit` (or equivalent) when splitting train/validation/test
for Model B. A random row-level split would risk the same vendor's
transactions appearing in both train and test, letting the model implicitly
learn vendor-specific patterns even without vendor identity as a feature —
undermining the entire premise of predicting for genuinely unseen vendors.

### 5.3.5 Export — Model B

In [62]:
export_path = "/content/drive/MyDrive/Colab Notebooks/Invoice-to-Payment Process Project/BPI Challenge 2019_Data/part2b_features.parquet"
model_b_features.to_parquet(export_path)

print(f"Exported: {model_b_features.shape}")
print(f"Columns: {list(model_b_features.columns)}")

Exported: (221242, 10)
Columns: ['item_category', 'spend_area', 'sub_spend_area', 'spend_classification', 'document_type', 'item_type', 'item_line_position', 'order_value', 'vendor', 'target_award_tier']


Final export: **221,242 transactions** × **10 columns** (order-level
attributes only: item category, spend area, sub-spend area, spend
classification, document type, item type, item line position, order
value, plus vendor identity and the 3-class Award tier target) — no
vendor-derived features, making this table applicable to genuinely new or
thin-history vendors, unlike Model A.

## 5.4 Key Findings Summary

**Summary of Key Findings — Feature Engineering:**

This notebook builds three feature tables for two predictive analyses —
Part 1 and Part 2 — trained and evaluated in Notebook 6.

**Shared foundation (5.1):** vendor tier ratings (Gold/Silver/Bronze/No
Award, per the UK's [Fair Payment Code](https://www.smallbusinesscommissioner.gov.uk/fpc/code-criteria/))
are computed using `ir_to_clear_days` — "days from receipt of invoice,"
matching the Fair Payment Code's actual definition. This includes 2-way
match cases, which lack a GR event but still have a valid IR timestamp. A
systematic column check confirmed which fields are genuinely case-constant.

**Part 1 — Case-Level Throughput Prediction:**

* **Scope:** `companyID_0000`, both 3-way match categories — 234,748
  total, 181,552 finished, cleaned cases (566 missing-GR + 88
  negative-duration cases dropped)
* **Prediction-point discipline:** strict `<` boundary (not `<=`) on each
  case's own GR timestamp, catching 200 same-timestamp batch artifacts
  that would otherwise leak
* **Cross-validated against [Rząd et al. (2019)](https://icpmconference.org/2019/wp-content/uploads/sites/6/2019/07/BPI-Challenge-Submission-2.pdf):**
  their top predictor, "Record Subsequent Invoice," is entirely absent
  from the pre-GR-filtered flags — concretely demonstrating it cannot be a
  legitimate forward-looking predictor from this prediction point
* **Export:** 181,552 cases × 25 columns

**Part 2 — Vendor Award Prediction:**

* **Scope:** all companies, both 3-way match categories plus 2-way match
  — 445 rateable vendors (1,222 Insufficient Data, 7 Not Applicable),
  drawn from 221,242 cases
* **Target:** No Award (233) / Bronze (160) / Silver+ (Silver 13 + Gold
  39 = 52) — Silver and Gold merged, since both are individually too
  small to model reliably
* **Two feature tables exported:** **Model A** (445 vendors × 24 columns
  — vendor-level aggregates: category mix, order value, exception rates,
  no outcome data, applicable to vendors with thin-but-nonzero history)
  and **Model B** (221,242 transactions × 10 columns — order-level
  attributes only, no vendor-derived features, applicable to genuinely
  new vendors)

**Conclusion:** The three feature tables are built on a shared, verified
foundation, with deliberately different temporal and structural logic —
**Part 1** follows strict prediction-point discipline for a forward-looking,
single-case forecast, while **Part 2** builds two complementary vendor-level
views for existing and new vendors respectively (Model A is vendor-based,
Model B is transaction-based). All three tables are exported for model
training in Notebook 6.

*Note:* For the methodological grounding of Part 2's Award tier and SME
threshold design (Fair Payment Code, World Bank/WEF citations, SPS
Commerce sample-size guidance), see the Methodological note in the
notebook's intro.

In [63]:
# Final cell: confirms a clean, complete run of this notebook

import datetime
from zoneinfo import ZoneInfo

cet_time = datetime.datetime.now(ZoneInfo("Europe/Berlin"))

print("=" * 50)
print(f"Notebook executed successfully, top to bottom.")
print(f"Run completed: {cet_time.strftime('%Y-%m-%d %H:%M %Z')}")
print("=" * 50)

Notebook executed successfully, top to bottom.
Run completed: 2026-09-14 20:10 CEST
